In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import os
import json
import glob
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import glob
import bitsandbytes as bnb

# Configuration
INPUT_DIR = "/kaggle/input/datasets/sayonsujitmondal/prompts/"
OUTPUT_DIR = "/kaggle/working/results/"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

# Range Parameters
START_FILE_INDEX = 9 
END_FILE_INDEX = 10 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created directory: {OUTPUT_DIR}")

# Get files
all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.jsonl")))
target_files = all_files[START_FILE_INDEX : END_FILE_INDEX + 1]

print(f"Found {len(all_files)} total files.")
print(f"Targeting {len(target_files)} files from index {START_FILE_INDEX} to {END_FILE_INDEX}.")

In [ ]:
# Verify Environment & List Target Files
print(f"--- Processing Batch: Indices {START_FILE_INDEX} to {END_FILE_INDEX} ---")

if not target_files:
    raise FileNotFoundError(f"No .jsonl files found in {INPUT_DIR}. Check your paths!")
else:
    print(f"Files to be processed ({len(target_files)} total):")
    for i, f in enumerate(target_files):
        print(f" [{START_FILE_INDEX + i}] {os.path.basename(f)}")

print("-" * 30)

# Verify Output Permissions
try:
    test_path = os.path.join(OUTPUT_DIR, 'permission_test.txt')
    with open(test_path, 'w') as f: f.write('test')
    os.remove(test_path)
    print(f"✓ Output directory writable: {OUTPUT_DIR}")
except Exception as e:
    raise PermissionError(f"CRITICAL ERROR: Cannot write to output path. {e}")

# Verify Hugging Face Authentication
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✓ Hugging Face authentication successful.")
    else:
        raise ValueError("HF_TOKEN not found in Kaggle Secrets!")
except Exception as e:
    raise RuntimeError(f"CRITICAL ERROR: Auth failed. {e}")

# Verify GPU Availability
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. Execution will be extremely slow.")
else:
    print(f"✓ GPU Detected: {torch.cuda.get_device_name(0)}")

print("--- All checks passed. Proceeding to Model Load ---")

In [ ]:
# Load Model & Define Inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)
print("✓ Model and Tokenizer loaded successfully!")

def generate_answer(prompt):
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=15, 
                do_sample=False, 
                pad_token_id=tokenizer.eos_token_id
            )
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return full_text[len(prompt):].strip()
    except Exception as e:
        return f"ERROR_IN_INFERENCE: {str(e)}"

In [ ]:
# Execution Loop
LIMIT_PER_FILE = 1200 

for file_path in target_files:
    file_name = os.path.basename(file_path)
    output_file = os.path.join(OUTPUT_DIR, f"results_{file_name}")
    
    print(f"\n>>> Processing: {file_name}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [json.loads(line) for line in f][:LIMIT_PER_FILE]
    
    with open(output_file, "w", encoding="utf-8") as out_f:
        for data in tqdm(lines, desc=f"File: {file_name}"):
            prediction = generate_answer(data["prompt"])
            
            result = {
                "id": data["id"],
                "gold_answer": data["answer"],
                "model_output": prediction,
                "source_file": file_name
            }
            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            
    print(f" Saved results to results_{file_name}")

print("\n" + "="*30 + "\nALL TASKS COMPLETE\n" + "="*30)